In [2]:
a = b"123;456;hola"
ip, port, msg = a.split(b";")
print(ip.decode(), port.decode(), msg.decode())

123 456 hola


In [18]:
_debug_ = True

def debug_print(message):
    """Prints debug messages if debugging is enabled."""
    if _debug_:
        print(f"[DEBUG] {message}")

def check_routes(routes_files_name: str, destinarion_address: tuple[str, int]) -> tuple[str, int]:
    """
    Función que revisa en orden la tabla de rutas para indicar la adirección del siguiente salto.
    """
    with open(routes_files_name, "r") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#"):
                continue  # Ignorar líneas vacías o comentarios
            parts = line.split(" ")  # el formato de la línea es:  RED I_port F_port next_IP next_PORT , por ej 127.0.0.1/24 8881 8882 127.0.0.1 8882

            if len(parts) != 5:
                debug_print(f"Línea de ruta inválida: {line}")
                continue  # Ignorar líneas mal formateadas
            dest_ip, initial_port, dest_port, next_hop_ip, next_hop_port = parts

            if (int(initial_port) <= destinarion_address[1] <= int(dest_port)):#dest_ip == destinarion_address[0] and 
                debug_print(f"Ruta encontrada para {destinarion_address}: siguiente salto {next_hop_ip}:{next_hop_port}")
                return (next_hop_ip, int(next_hop_port))
    debug_print(f"No se encontró ruta para {destinarion_address}")
    return None

In [3]:
def parse_packet(IP_BRAcKETS: bytes) -> dict:
    """
    Función que permite extraer los headers y datos del paquete 
    recibido, transformando a una estructura de datos conveniente
    """
    ip, port, ttl, msg = IP_BRAcKETS.split(b";")
    structured_data = {
        "ip": ip.decode(),
        "port": int(port.decode()),
        "msg": msg.decode(),
        "ttl": int(ttl.decode())
    }

    return structured_data

def create_packet(structured_data: dict) -> bytes:
    """
    Función que permite crear un paquete a partir de los headers y datos en una estructura de datos conveniente
    """
    ip = structured_data["ip"]
    port = structured_data["port"]
    msg = structured_data["msg"]
    ttl = structured_data.get("ttl", 64)  # Valor por defecto de TTL si no se proporciona
    packet = f"{ip};{port};{ttl};{msg}".encode()
    return packet

In [4]:
for IP_packet_v1 in [
    b"127.0.0.1;8883;10;hola",
    b"127.0.0.1;8885;64;Hello, world!",
    b"127.0.0.1;8883;1;hola",
]:
    parsed_IP_packet = parse_packet(IP_packet_v1)
    IP_packet_v2 = create_packet(parsed_IP_packet)
    print("IP_packet_v1 == IP_packet_v2 ? {}".format(IP_packet_v1 == IP_packet_v2))

IP_packet_v1 == IP_packet_v2 ? True
IP_packet_v1 == IP_packet_v2 ? True
IP_packet_v1 == IP_packet_v2 ? True


In [5]:
a = b"127.0.0.1;8883;10;asdasd"
a.split(b";", 3)

[b'127.0.0.1', b'8883', b'10', b'asdasd']